# Scientific Paper Knowledge Graph with Embeddings and Local LLMs

Course project pipeline for building a domain-oriented knowledge graph from scientific PDF papers.

Final graphs:

1. Embedding Similarity Graph
2. LLM Relation Graph
3. Weighted Hybrid Knowledge Graph


In [35]:
# 1. Install dependencies
# In Google Colab, run this cell if packages are missing.
# !pip install -q pandas numpy PyMuPDF sentence-transformers scikit-learn networkx pyvis matplotlib transformers torch accelerate tqdm
!git pull https://github.com/Voisan/knowledge_graph

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 415 bytes | 415.00 KiB/s, done.
From https://github.com/Voisan/knowledge_graph
 * branch            HEAD       -> FETCH_HEAD
Updating e996421..085fcca
Fast-forward
 src/visualization.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)


In [36]:
%cd /content/knowledge_graph

import sys
sys.path.insert(0, "/content/knowledge_graph")

/content/knowledge_graph


In [37]:
!pip install pyvis pymupdf

In [38]:
# 2. Configuration
from pathlib import Path
import sys
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

PROJECT_ROOT = Path("/content/knowledge_graph")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PAPERS_PATH = PROJECT_ROOT / "data" / "papers"
RESULTS_PATH = PROJECT_ROOT / "data" / "results"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print("Current dir:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Papers path:", PAPERS_PATH)
print("Papers path exists:", PAPERS_PATH.exists())


EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SIMILARITY_THRESHOLD = 0.55
TOP_K = 5
LLM_SIMILARITY_THRESHOLD = 0.50
LLM_TOP_K = 10
LLM_MAX_PAIRS = None
USE_LLM = True
HYBRID_ALPHA = 0.5

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Papers path:", PAPERS_PATH)
print("Results path:", RESULTS_PATH)


Current dir: /content/knowledge_graph
Project root: /content/knowledge_graph
Papers path: /content/knowledge_graph/data/papers
Papers path exists: True
Project root: /content/knowledge_graph
Papers path: /content/knowledge_graph/data/papers
Results path: /content/knowledge_graph/data/results


In [39]:
# 3. Imports
import networkx as nx
import pandas as pd
from tqdm.auto import tqdm

from src.pdf_parser import parse_papers_from_folder
from src.preprocessing import clean_text, truncate_text
from src.embeddings import (
    load_embedding_model,
    compute_embeddings,
    compute_similarity_matrix,
    get_candidate_pairs,
)
from src.llm_relation_extractor import load_llm, classify_relation
from src.graph_builder import (
    build_similarity_graph,
    filter_llm_relations,
    build_llm_relation_graph,
    build_weighted_hybrid_graph,
    graph_to_nodes_df,
    graph_to_edges_df,
)
from src.graph_metrics import (
    compare_graphs,
    get_communities_df,
    get_community_summary_df,
    calculate_centrality_df,
    explain_path,
    find_paper_id_by_filename,
)
from src.visualization import visualize_graph_pyvis, plot_metric_comparison
from src.utils import save_dataframe, save_graph_graphml


In [40]:
# 4-6. Parse PDFs, clean text, and optionally fix titles
papers_df = parse_papers_from_folder(PAPERS_PATH)

TITLE_MAP = {
    # "qlora.pdf": "QLoRA: Efficient Finetuning of Quantized LLMs",
    # "BERT.pdf": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
}
if not papers_df.empty:
    papers_df["title"] = papers_df.apply(
        lambda row: TITLE_MAP.get(str(row["filename"]), row["title"]),
        axis=1,
    )
    papers_df["text_clean"] = papers_df["text"].fillna("").apply(clean_text)
    papers_df["text_for_embedding"] = papers_df.apply(
        lambda row: truncate_text(
            row["abstract"] if isinstance(row["abstract"], str) and row["abstract"].strip() else row["text_clean"],
            5000,
        ),
        axis=1,
    )
else:
    papers_df["text_clean"] = []
    papers_df["text_for_embedding"] = []

save_dataframe(papers_df, RESULTS_PATH / "papers.csv")
print(f"Parsed papers: {len(papers_df)}")
display(papers_df[["paper_id", "filename", "title", "abstract"]].head(20))


Parsing PDFs:   0%|          | 0/20 [00:00<?, ?it/s]

Parsed papers: 20


,paper_id,filename,title,abstract
0,0,BERT.pdf,BERT: Pre-training of Deep Bidirectional Trans...,We introduce a new language representation mod...
1,1,albert.pdf,Published as a conference paper at ICLR 2020,Increasing model size when pretraining natural...
2,2,alluneed.pdf,"Provided proper attribution is provided, Googl...",The dominant sequence transduction models are ...
3,3,distilbert.pdf,"DistilBERT, a distilled version of BERT: smaller,",As Transfer Learning from large-scale pre-trai...
4,4,extracknowgraphs.pdf,KGGEN: EXTRACTING KNOWLEDGE GRAPHS FROM,Recent interest in building foundation models ...
5,5,firstgpt.pdf,Improving Language Understanding,Natural language understanding comprises a wid...
6,6,humanfeedback.pdf,Training language models to follow instructions,Making language models bigger does not inheren...
7,7,knowgraph.pdf,Large Language Models Meet Knowledge Graphs to,"Recently, it has been shown that the incorpora..."
8,8,knowgraphllm.pdf,Published as a conference paper at ICAIS 2025,Knowledge Graphs (KGs) have long served as a f...
9,9,linformer.pdf,Linformer: Self-Attention with Linear Complexity,Large transformer models have shown extraordin...


In [41]:
# 7-9. Embeddings, similarity matrix, and candidate pairs
embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)
embeddings = compute_embeddings(papers_df["text_for_embedding"].tolist(), embedding_model)
similarity_matrix = compute_similarity_matrix(embeddings)

candidate_pairs = get_candidate_pairs(similarity_matrix, threshold=SIMILARITY_THRESHOLD, top_k=TOP_K)
llm_candidate_pairs = get_candidate_pairs(similarity_matrix, threshold=LLM_SIMILARITY_THRESHOLD, top_k=LLM_TOP_K)

candidate_pairs_df = pd.DataFrame(candidate_pairs, columns=["source", "target", "similarity"])
save_dataframe(candidate_pairs_df, RESULTS_PATH / "candidate_pairs.csv")

print(f"Embedding graph candidate pairs: {len(candidate_pairs)}")
print(f"LLM candidate pairs: {len(llm_candidate_pairs)}")
display(candidate_pairs_df.head())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding graph candidate pairs: 52
LLM candidate pairs: 93


,source,target,similarity
0,0,1,0.763259
1,0,2,0.707575
2,0,3,0.695212
3,0,5,0.742592
4,0,7,0.599577


In [42]:
# 10. Embedding Similarity Graph
embedding_graph = build_similarity_graph(papers_df, candidate_pairs)
save_graph_graphml(embedding_graph, RESULTS_PATH / "embedding_graph.graphml")
print("Embedding Similarity Graph:", embedding_graph)


Embedding Similarity Graph: Graph with 20 nodes and 52 edges


In [43]:
# 11-13. Load local LLM and extract typed relations
LLM_COLUMNS = [
    "source",
    "target",
    "relation",
    "confidence",
    "reason",
    "candidate_similarity",
    "source_method",
]
llm_relations_raw = []

if USE_LLM and llm_candidate_pairs:
    llm_model, tokenizer = load_llm(LLM_MODEL_NAME)
    papers_by_id = papers_df.set_index("paper_id").to_dict(orient="index")
    pairs_for_llm = llm_candidate_pairs[:LLM_MAX_PAIRS] if LLM_MAX_PAIRS else llm_candidate_pairs

    for pair in tqdm(pairs_for_llm, desc="LLM relation extraction"):
        paper_a = {"paper_id": int(pair["source"]), **papers_by_id[int(pair["source"])]}
        paper_b = {"paper_id": int(pair["target"]), **papers_by_id[int(pair["target"])]}
        result = classify_relation(paper_a, paper_b, llm_model, tokenizer)
        result["candidate_similarity"] = float(pair["similarity"])
        result["source_method"] = "llm"
        llm_relations_raw.append(result)
else:
    print("Local LLM extraction is disabled or there are no candidate pairs.")

llm_relations_raw_df = pd.DataFrame(llm_relations_raw)
for column in LLM_COLUMNS:
    if column not in llm_relations_raw_df.columns:
        llm_relations_raw_df[column] = ""
llm_relations_raw_df = llm_relations_raw_df[LLM_COLUMNS]
save_dataframe(llm_relations_raw_df, RESULTS_PATH / "llm_relations_raw.csv")

print(f"Raw LLM relation rows: {len(llm_relations_raw_df)}")
display(llm_relations_raw_df.head())


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM relation extraction:   0%|          | 0/93 [00:00<?, ?it/s]

Raw LLM relation rows: 93


,source,target,relation,confidence,reason,candidate_similarity,source_method
0,0,1,EXTENDS,0.90,"Both papers discuss improvements over BERT, sp...",0.763259,llm
1,0,2,NO_RELATION,0.00,The papers discuss different topics (natural l...,0.707575,llm
2,0,3,EXTENDS,0.90,Both papers discuss improvements over existing...,0.695212,llm
3,0,5,COMPARES_WITH,0.95,Both papers discuss improvements in language u...,0.742592,llm
4,0,6,COMPARES_WITH,0.90,Both papers discuss the alignment of language ...,0.590280,llm


In [44]:
# 14-16. Filter LLM edges and build LLM Relation Graph
llm_relations_filtered_df = filter_llm_relations(llm_relations_raw_df)
save_dataframe(llm_relations_filtered_df, RESULTS_PATH / "llm_relations_filtered.csv")

llm_graph = build_llm_relation_graph(papers_df, llm_relations_filtered_df)
save_graph_graphml(llm_graph, RESULTS_PATH / "llm_graph.graphml")

print(f"Filtered LLM relation rows: {len(llm_relations_filtered_df)}")
print("LLM Relation Graph:", llm_graph)
display(llm_relations_filtered_df.head())


Filtered LLM relation rows: 40
LLM Relation Graph: Graph with 20 nodes and 40 edges


,source,target,relation,confidence,reason,candidate_similarity,source_method
0,0,1,EXTENDS,0.90,"Both papers discuss improvements over BERT, sp...",0.763259,llm
1,0,3,EXTENDS,0.90,Both papers discuss improvements over existing...,0.695212,llm
2,0,5,COMPARES_WITH,0.95,Both papers discuss improvements in language u...,0.742592,llm
3,0,6,COMPARES_WITH,0.90,Both papers discuss the alignment of language ...,0.590280,llm
4,0,7,COMPARES_WITH,0.90,Both papers discuss advancements in using larg...,0.599577,llm


In [45]:
# 17. Weighted Hybrid Knowledge Graph
if USE_LLM:
    hybrid_graph = build_weighted_hybrid_graph(
        papers_df,
        embedding_graph,
        llm_relations_filtered_df,
        alpha=HYBRID_ALPHA,
    )
else:
    hybrid_graph = embedding_graph.copy()

save_graph_graphml(hybrid_graph, RESULTS_PATH / "hybrid_graph.graphml")
save_dataframe(graph_to_nodes_df(hybrid_graph), RESULTS_PATH / "hybrid_nodes.csv")
save_dataframe(graph_to_edges_df(hybrid_graph), RESULTS_PATH / "hybrid_edges.csv")

print("Weighted Hybrid Knowledge Graph:", hybrid_graph)
display(graph_to_edges_df(hybrid_graph).head())


Weighted Hybrid Knowledge Graph: Graph with 20 nodes and 66 edges


,source,target,relation,weight,confidence,candidate_similarity,source_method,reason
0,0,1,EXTENDS,0.831629,0.90,0.763259,llm,"Both papers discuss improvements over BERT, sp..."
1,0,3,EXTENDS,0.797606,0.90,0.695212,llm,Both papers discuss improvements over existing...
2,0,5,COMPARES_WITH,0.846296,0.95,0.742592,llm,Both papers discuss improvements in language u...
3,0,6,COMPARES_WITH,0.745140,0.90,0.590280,llm,Both papers discuss the alignment of language ...
4,0,7,COMPARES_WITH,0.749788,0.90,0.599577,llm,Both papers discuss advancements in using larg...


In [46]:
# 18. Graph metrics
if USE_LLM:
    graphs = {
        "Embedding Similarity Graph": embedding_graph,
        "LLM Relation Graph": llm_graph,
        "Weighted Hybrid Knowledge Graph": hybrid_graph,
    }
else:
    graphs = {
        "Embedding Similarity Graph": embedding_graph,
        "Weighted Hybrid Knowledge Graph": hybrid_graph,
    }

metrics_df = compare_graphs(graphs)
save_dataframe(metrics_df, RESULTS_PATH / "graph_metrics.csv")

display(metrics_df)


,graph,number_of_nodes,number_of_edges,density,number_connected_components,average_degree,average_clustering,number_of_communities
0,Embedding Similarity Graph,20,52,0.273684,2,5.2,0.433946,4
1,LLM Relation Graph,20,40,0.210526,3,4.0,0.266314,5
2,Weighted Hybrid Knowledge Graph,20,66,0.347368,2,6.6,0.445086,4


In [47]:
# 19. Communities for the hybrid graph
communities_df = get_communities_df(hybrid_graph)
community_summary_df = get_community_summary_df(communities_df)

save_dataframe(communities_df, RESULTS_PATH / "communities.csv")
save_dataframe(community_summary_df, RESULTS_PATH / "community_summary.csv")

print(f"Communities: {community_summary_df.shape[0]}")
display(community_summary_df)


Communities: 4


,community_id,papers_count,papers
0,0,9,BERT: Pre-training of Deep Bidirectional Trans...
1,1,6,"Published as a conference paper at ICLR 2020, ..."
2,2,4,Training language models to follow instruction...
3,3,1,Published as a conference paper at ICAIS 2025


In [48]:
# 20. Centrality for the hybrid graph
centrality_df = calculate_centrality_df(hybrid_graph)
save_dataframe(centrality_df, RESULTS_PATH / "centrality.csv")

display(centrality_df.head(10))


,paper_id,title,filename,degree_centrality,betweenness_centrality,pagerank
0,0,BERT: Pre-training of Deep Bidirectional Trans...,BERT.pdf,0.684211,0.116959,0.092182
1,1,Published as a conference paper at ICLR 2020,albert.pdf,0.578947,0.087719,0.079122
2,5,Improving Language Understanding,firstgpt.pdf,0.578947,0.023392,0.078908
3,15,Retrieval-Augmented Generation for,rag.pdf,0.578947,0.011696,0.077651
4,3,"DistilBERT, a distilled version of BERT: smaller,",distilbert.pdf,0.421053,0.134503,0.065640
5,13,Longformer: The Long-Document Transformer,longformer.pdf,0.473684,0.005848,0.063820
6,6,Training language models to follow instructions,humanfeedback.pdf,0.473684,0.146199,0.062210
7,17,RoBERTa: A Robustly Optimized BERT Pretraining...,roberta.pdf,0.421053,0.035088,0.057518
8,19,"SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND",selfrag.pdf,0.421053,0.017544,0.057378
9,2,"Provided proper attribution is provided, Googl...",alluneed.pdf,0.421053,0.005848,0.055595


In [49]:
# 21. Shortest path explanation example
path_example_df = pd.DataFrame()
path_candidates = [
    ("qlora", "BERT"),
    ("selfrag", "rag"),
]

for source_part, target_part in path_candidates:
    try:
        source_id = find_paper_id_by_filename(papers_df, source_part)
        target_id = find_paper_id_by_filename(papers_df, target_part)
        path_example_df = explain_path(hybrid_graph, source_id, target_id)
        if not path_example_df.empty:
            print(f"Path example: {source_part} -> {target_part}")
            break
    except ValueError as exc:
        print(exc)

if path_example_df.empty and hybrid_graph.number_of_edges() > 0:
    component = max(nx.connected_components(hybrid_graph), key=len)
    nodes = list(component)
    if len(nodes) >= 2:
        path_example_df = explain_path(hybrid_graph, int(nodes[0]), int(nodes[-1]))
        print(f"Fallback path example: {nodes[0]} -> {nodes[-1]}")

save_dataframe(path_example_df, RESULTS_PATH / "path_example.csv")
display(path_example_df)


Path example: qlora -> BERT


,step,source_id,source_title,relation,target_id,target_title,weight,source_method,reason
0,1,14,QLORA: Efficient Finetuning of Quantized LLMs,SIMILAR_TO,1,Published as a conference paper at ICLR 2020,0.575966,embedding,Edge added by embedding similarity.
1,2,1,Published as a conference paper at ICLR 2020,EXTENDS,0,BERT: Pre-training of Deep Bidirectional Trans...,0.831629,llm,"Both papers discuss improvements over BERT, sp..."


In [50]:
# 22. Optional visualization
try:
    visualize_graph_pyvis(embedding_graph, str(RESULTS_PATH / "embedding_similarity_graph.html"))
    if USE_LLM:
        visualize_graph_pyvis(llm_graph, str(RESULTS_PATH / "llm_relation_graph.html"))
    visualize_graph_pyvis(hybrid_graph, str(RESULTS_PATH / "hybrid_graph.html"))

    for metric_name in ["number_of_edges", "density", "average_degree", "average_clustering", "number_of_communities"]:
        plot_metric_comparison(metrics_df, metric_name, str(RESULTS_PATH / f"compare_{metric_name}.png"))
    print("Visualizations saved to", RESULTS_PATH)
except Exception as exc:
    print("Visualization skipped:", exc)


Visualizations saved to /content/knowledge_graph/data/results


In [51]:
# 23. Export summary
expected_files = [
    "papers.csv",
    "candidate_pairs.csv",
    "llm_relations_raw.csv",
    "llm_relations_filtered.csv",
    "graph_metrics.csv",
    "communities.csv",
    "community_summary.csv",
    "centrality.csv",
    "path_example.csv",
    "hybrid_nodes.csv",
    "hybrid_edges.csv",
    "embedding_graph.graphml",
    "llm_graph.graphml",
    "hybrid_graph.graphml",
]

for file_name in expected_files:
    path = RESULTS_PATH / file_name
    print(f"{file_name}: {'OK' if path.exists() else 'missing'}")


papers.csv: OK
candidate_pairs.csv: OK
llm_relations_raw.csv: OK
llm_relations_filtered.csv: OK
graph_metrics.csv: OK
communities.csv: OK
community_summary.csv: OK
centrality.csv: OK
path_example.csv: OK
hybrid_nodes.csv: OK
hybrid_edges.csv: OK
embedding_graph.graphml: OK
llm_graph.graphml: OK
hybrid_graph.graphml: OK


In [52]:
!zip -r /content/results.zip /content/knowledge_graph/data/results
from google.colab import files
files.download("/content/results.zip")

updating: content/knowledge_graph/data/results/ (stored 0%)
updating: content/knowledge_graph/data/results/embedding_graph.graphml (deflated 88%)
updating: content/knowledge_graph/data/results/centrality.csv (deflated 51%)
updating: content/knowledge_graph/data/results/llm_relations_raw.csv (deflated 76%)
updating: content/knowledge_graph/data/results/candidate_pairs.csv (deflated 51%)
updating: content/knowledge_graph/data/results/hybrid_graph.html (deflated 75%)
updating: content/knowledge_graph/data/results/hybrid_edges.csv (deflated 72%)
updating: content/knowledge_graph/data/results/communities.csv (deflated 46%)
updating: content/knowledge_graph/data/results/compare_density.png (deflated 19%)
updating: content/knowledge_graph/data/results/path_example.csv (deflated 37%)
updating: content/knowledge_graph/data/results/compare_average_clustering.png (deflated 18%)
updating: content/knowledge_graph/data/results/compare_average_degree.png (deflated 18%)
updating: content/knowledge_gra

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
!zip -r /content/processed.zip /content/knowledge_graph/data/processed
from google.colab import files
files.download("/content/processed.zip")

updating: content/knowledge_graph/data/processed/ (stored 0%)
updating: content/knowledge_graph/data/processed/.gitkeep (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretation

Embedding Graph gives broad coverage and connectivity.

LLM Relation Graph gives typed, interpretable semantic links.

Weighted Hybrid Knowledge Graph combines both approaches and is the final domain-oriented knowledge graph for analysis and defense.
